# CHAT ENGINE - Used for having a multi-turn conversation with your data while considering previous conversation context.

| Chat Engine                       | Purpose                                                                                             |
| --------------------------------- | --------------------------------------------------------------------------------------------------- |
| **SimpleChatEngine**              | Basic conversational chat without retrieval                                                         |
| **ContextChatEngine**             | Chat with retrieved context from your index                                                         |
| **CondenseQuestionChatEngine**    | Converts follow-up questions into standalone questions before retrieval                             |
| **CondensePlusContextChatEngine** | Condenses the conversation into a standalone question, retrieves context, then generates the answer |
| **CondenseQuestionChatEngine**    | Useful when questions depend heavily on previous conversation                                       |
| **ContextChatEngine**             | Useful for chatting directly with your indexed documents                                            |


<div align="center">
    <img src="images/chatengine.png" width="700" height="2000">
</div>

In [1]:
from dotenv import load_dotenv
from llama_index.llms.openai import OpenAI

load_dotenv()
llm = OpenAI(model="gpt-4o-mini")

In [2]:
import logging

logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("llama_index").setLevel(logging.ERROR)
logging.getLogger("openai").setLevel(logging.ERROR)
logging.disable(logging.CRITICAL)

# 1. Simple Chat Engine

In [3]:
from llama_index.core.chat_engine import SimpleChatEngine

chat_engine = SimpleChatEngine.from_defaults(llm=llm)
question = input("You: ")
response = chat_engine.chat(question)
print("\nAI:", response)

You:  What is AI



AI: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. AI systems can perform tasks that typically require human intelligence, such as understanding natural language, recognizing patterns, solving problems, and making decisions.

AI can be categorized into several types:

1. **Narrow AI**: Also known as weak AI, this type of AI is designed to perform a specific task, such as voice recognition, image classification, or playing chess. Most AI applications today fall into this category.

2. **General AI**: Also known as strong AI or AGI (Artificial General Intelligence), this type of AI would have the ability to understand, learn, and apply intelligence across a wide range of tasks, similar to a human being. As of now, general AI remains largely theoretical and has not yet been achieved.

3. **Machine Learning (ML)**: A subset of AI that involves training algorithms on data so that they can make pred

# 2. Context Chat Engine - Directly uses the current question + retrieved context.

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

documents = SimpleDirectoryReader(input_files=["data/HopeAI.pdf"]).load_data()

index = VectorStoreIndex.from_documents(documents)

chat_engine = index.as_chat_engine(
    chat_mode="context",
    llm=OpenAI(
        model="gpt-5-nano",
        temperature=0
    )
)
while True:
    question = input("You: ")
    if question.lower() == "exit":
        break
    response = chat_engine.chat(question)
    print("AI:", response)

# 3. Condense Question Chat Engine - Uses the conversation history to rewrite the follow-up question into a standalone question, then retrieves information for that question.

In [4]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# Load PDF
documents = SimpleDirectoryReader(input_files=["data/HopeAI.pdf"]).load_data()

# Create index
index = VectorStoreIndex.from_documents(documents)

# Create Condense Question Chat Engine
chat_engine = index.as_chat_engine(
    chat_mode="condense_question",
    llm=llm
)

# Chat
while True:
    question = input("You: ")
    if question.lower() == "exit":
        break
    response = chat_engine.chat(question)
    print("AI:", response)

C:\Users\User\anaconda3\envs\ai\Lib\site-packages\pydantic\_internal\_generate_schema.py:937: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `BasePromptTemplate` to V2.
  warnings.warn(


You:  What courses does Hope AI offer?


AI: Hope AI offers courses in Artificial Intelligence and Data Science, covering topics such as Classification, Regression, Clustering, Deep Learning, Time Series Analysis, and Natural Language Processing (NLP). Additionally, courses include advanced AI topics like Transformer Architecture, Large Language Models (LLM), AI Agents, and Retrieval-Augmented Generation (RAG). The platform also provides practical training with real-time application deployment.


You:  How long is the course


AI: The duration of the courses offered is 2 hours per day for 6 months.


You:  exit


# 4. Condense Plus Context Chat Engine - Condenses the question and uses retrieved context

In [5]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.llms.openai import OpenAI

# 1. Load PDF
documents = SimpleDirectoryReader(
    input_files=["data/HopeAI.pdf"]
).load_data()

# 2. Create index
index = VectorStoreIndex.from_documents(documents)

# 3. Create Condense Plus Context Chat Engine
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",
    llm=OpenAI(
        model="gpt-5-nano",
        temperature=0
    )
)

# 4. Chat
while True:
    question = input("You: ")

    if question.lower() == "exit":
        break

    response = chat_engine.chat(question)

    print("AI:", response)

You:  What courses does Hope AI offer?


AI: Hope AI offers AI and Data Science courses covering:

- Classification, Regression, Clustering
- Deep Learning
- Time Series Analysis
- Natural Language Processing (NLP)
- Cloud and DevOps (AWS, GCP, Azure)
- Advanced AI Topics (Transformer Architecture, Large Language Models [LLMs], AI Agents, Retrieval-Augmented Generation [RAG])

These topics are taught with real-time deployment applications, and the courses are self-paced with lifelong access.


You:  How long is the course


AI: The course lasts 6 months, with sessions of 2 hours per day.


You:  Who is the Tutor of Hope AI?


AI: The tutor is Ramisha Rani K, who has 8+ years of experience in AI and Data Science and specializes in Python, Machine Learning, NLP, Time Series Analysis, Deep Learning, and ChatGPT API.


You:  exit
